[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_2_3/11_tag_2_3_optimierung_lr_scheduler_early_stopping.ipynb)

# Tag 2/3 - 11 Optimierung, Lernratenpläne und Callbacks

Der Optimizer wird in `compile` gesetzt. Er bestimmt, wie Gradienten in Parameterupdates übersetzt werden.

Dieses Notebook trennt drei Dinge:

1. Optimizer-Vergleich
2. Callbacks allgemein
3. Learning-Rate-Scheduler und Early Stopping als konkrete Callback-Beispiele


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Setup abgeschlossen.")


In [ ]:
X, y = make_moons(n_samples=1000, noise=0.28, random_state=RANDOM_STATE)
X = StandardScaler().fit_transform(X).astype("float32")
y = y.astype("float32").reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y.ravel()
)


def build_model(width=32):
    return keras.Sequential(
        [
            keras.Input(shape=(2,), name="features"),
            layers.Dense(width, activation="relu", name="hidden_1"),
            layers.Dense(width, activation="relu", name="hidden_2"),
            layers.Dense(1, activation="sigmoid", name="output"),
        ]
    )


## Optimizer-Vergleich


In [ ]:
optimizer_factories = {
    "SGD": lambda: keras.optimizers.SGD(learning_rate=0.05),
    "SGD + Momentum": lambda: keras.optimizers.SGD(learning_rate=0.05, momentum=0.9),
    "RMSprop": lambda: keras.optimizers.RMSprop(learning_rate=0.01),
    "Adam": lambda: keras.optimizers.Adam(learning_rate=0.01),
}

histories = {}
for name, optimizer_factory in optimizer_factories.items():
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = build_model()
    model.compile(optimizer=optimizer_factory(), loss="binary_crossentropy", metrics=["accuracy"])
    history = model.fit(X_train, y_train, validation_split=0.25, epochs=40, batch_size=32, verbose=0)
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    histories[name] = history
    print(f"{name:>14}: Test Accuracy={test_accuracy:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in histories.items():
    axes[0].plot(history.history["val_loss"], label=name)
    axes[1].plot(history.history["val_accuracy"], label=name)
axes[0].set_title("Validation Loss")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoche")
axes[1].legend()
plt.tight_layout()
plt.show()


## Was sind Callbacks?

Ein Callback ist Code, den Keras während des Trainings automatisch an bestimmten Zeitpunkten aufruft, zum Beispiel:

- vor oder nach einer Epoche
- vor oder nach einem Batch
- am Trainingsanfang oder Trainingsende

Callbacks ändern nicht die Modellarchitektur. Sie beobachten oder steuern den Trainingsprozess. Typische Beispiele sind:

- `LearningRateScheduler`: setzt die Lernrate pro Epoche.
- `ReduceLROnPlateau`: senkt die Lernrate, wenn sich `val_loss` nicht verbessert.
- `EarlyStopping`: beendet Training, wenn keine Verbesserung mehr kommt.
- eigene Callback-Klassen zum Loggen zusätzlicher Werte.


In [ ]:
class EpochLogger(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if epoch in [0, 9, 19]:
            print(
                f"Epoche {epoch + 1:02d}: "
                f"loss={logs.get('loss'):.3f}, "
                f"val_loss={logs.get('val_loss'):.3f}, "
                f"val_accuracy={logs.get('val_accuracy'):.1%}"
            )


tf.keras.utils.set_random_seed(RANDOM_STATE)
logger_model = build_model()
logger_model.compile(optimizer=keras.optimizers.Adam(0.01), loss="binary_crossentropy", metrics=["accuracy"])
logger_model.fit(
    X_train,
    y_train,
    validation_split=0.25,
    epochs=20,
    batch_size=32,
    verbose=0,
    callbacks=[EpochLogger()],
);


## Learning-Rate-Scheduler: Warmup + Cosine

Die Lernrate startet klein, steigt in den ersten Epochen an und fällt danach weich ab. Das ist eine Optimierungsstrategie, die als Callback umgesetzt wird.


In [ ]:
def warmup_cosine(epoch, current_lr):
    warmup_epochs = 5
    total_epochs = 80
    min_lr = 0.001
    max_lr = 0.03
    if epoch < warmup_epochs:
        return min_lr + (max_lr - min_lr) * (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + np.cos(np.pi * progress))


class LrHistory(callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.values = []

    def on_epoch_end(self, epoch, logs=None):
        self.values.append(float(tf.keras.backend.get_value(self.model.optimizer.learning_rate)))


tf.keras.utils.set_random_seed(RANDOM_STATE)
scheduled_model = build_model()
scheduled_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])

lr_history = LrHistory()
schedule_history = scheduled_model.fit(
    X_train,
    y_train,
    validation_split=0.25,
    epochs=80,
    batch_size=32,
    verbose=0,
    callbacks=[callbacks.LearningRateScheduler(warmup_cosine), lr_history],
)

test_loss, test_accuracy = scheduled_model.evaluate(X_test, y_test, verbose=0)
print(f"Trainierte Epochen: {len(schedule_history.history['loss'])}")
print(f"Test Accuracy: {test_accuracy:.1%}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(schedule_history.history["val_loss"], label="validation")
axes[0].set_title("Validation Loss")
axes[0].set_xlabel("Epoche")
axes[1].plot(schedule_history.history["val_accuracy"], label="validation", color="green")
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoche")
axes[2].plot(lr_history.values)
axes[2].set_title("Learning Rate")
axes[2].set_xlabel("Epoche")
plt.tight_layout()
plt.show()


## Early Stopping Callback: Training mit und ohne Early Stopping

Early Stopping ist selbst ein Callback. Es beobachtet während `fit(...)` eine Metrik, hier `val_loss`, und beendet das Training, wenn sich diese Metrik mehrere Epochen nicht verbessert.

Der Vergleich ist deshalb: ein normaler Trainingslauf über 90 Epochen gegen einen Trainingslauf mit `callbacks.EarlyStopping(...)`. In beiden Fällen schauen wir auf Validation Loss, Validation Accuracy und Test Accuracy.


In [ ]:
def build_overfit_model():
    return build_model(width=96)


tf.keras.utils.set_random_seed(RANDOM_STATE)
no_early_model = build_overfit_model()
no_early_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss="binary_crossentropy", metrics=["accuracy"])
no_early_history = no_early_model.fit(
    X_train,
    y_train,
    validation_split=0.25,
    epochs=90,
    batch_size=32,
    verbose=0,
)
no_early_test_loss, no_early_test_accuracy = no_early_model.evaluate(X_test, y_test, verbose=0)

tf.keras.utils.set_random_seed(RANDOM_STATE)
early_model = build_overfit_model()
early_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss="binary_crossentropy", metrics=["accuracy"])
early = callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
early_history = early_model.fit(
    X_train,
    y_train,
    validation_split=0.25,
    epochs=90,
    batch_size=32,
    verbose=0,
    callbacks=[early],
)
early_test_loss, early_test_accuracy = early_model.evaluate(X_test, y_test, verbose=0)

print(f"Ohne Early Stopping: Epochen=90, Test Accuracy={no_early_test_accuracy:.1%}")
print(f"Mit Early Stopping:  Epochen={len(early_history.history['loss'])}, Test Accuracy={early_test_accuracy:.1%}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(no_early_history.history["val_loss"], label="ohne Early Stopping")
axes[0].plot(early_history.history["val_loss"], label="mit Early Stopping")
axes[0].set_title("Validation Loss")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[1].plot(no_early_history.history["val_accuracy"], label="ohne Early Stopping")
axes[1].plot(early_history.history["val_accuracy"], label="mit Early Stopping")
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoche")
axes[1].legend()
axes[2].bar(["ohne", "mit"], [no_early_test_accuracy, early_test_accuracy])
axes[2].set_ylim(0.0, 1.0)
axes[2].set_title("Test Accuracy")
plt.tight_layout()
plt.show()
